#**Proyecto Integrador de Análisis de Datos**



In [ ]:
# Módulo 6 - Proyecto Integrador de Análisis de Datos

# Importar librerías
#------------------------------------------
# Importación de librerías esenciales para datos y archivos en Colab
from google.colab import files
import pandas as pd
import numpy as np

print('\n Cargar base datos')


subido = files.upload()                   # Seleccionar archivo
nombre_archivo = list(subido.keys())[0]   # Obtener el nombre
# Cargamos con utf-8 y usamos encoding_errors='replace' para manejar caracteres no decodificables
datos_base = pd.read_csv(nombre_archivo, encoding='utf-8', encoding_errors='replace')  # Leer el CSV

datos_ordenes = pd.DataFrame(datos_base) # DataFrame
datos_ordenes.head(10)                   # Mostrar tabla

datos_ordenes.dtypes

#Exploración de datos
#-------------------------------------
# Conteo de valores nulos
print("\n                     Valores nulos                         ")
valores_nulos = datos_ordenes.isnull().sum()
print(valores_nulos)

# Revisión de países únicos
print("                     Valores unicos de paises                           ")
paises = sorted(datos_ordenes['country'].unique())
print(f'Cantidad de paises unicos: {len(paises)}')
print(f'Listado de paises: \n{paises}')

# Revisión de categorías únicas
print("\n                     Valores unicos de categorias                           ")
categorias = datos_ordenes['category'].unique()
print(categorias)

# Identificación de descuentos negativos (inválidos)
print("\n                     Valores descuento de invalidos                           ")
descuentos_invalidos = datos_ordenes[datos_ordenes['discount'] < 0]['discount'] # Se muestran los descuentos invalidos
print(descuentos_invalidos)

#Limpieza de datos
#-------------------------------------

# Estandarización de nombres de columnas (minúsculas, sin espacios ni símbolos)
datos_ordenes.columns = (
    datos_ordenes.columns.str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
    .str.replace('[^a-z0-9_]', '', regex=True)
)

#Limpiar espacios en blanco alrededor de textos
#-------------------------------------
# Eliminar espacios innecesarios al inicio y final de columnas de texto
text_cols = datos_ordenes.select_dtypes(include='object').columns
for col in text_cols:
    datos_ordenes[col] = datos_ordenes[col].astype(str).str.strip()

# Formato de título (Mayúscula inicial) para textos categóricos
datos_ordenes['country'] = datos_ordenes['country'].str.title()
datos_ordenes['customer_name'] = datos_ordenes['customer_name'].str.title()
datos_ordenes['state'] = datos_ordenes['state'].str.title()

#Estandarizacion de textos de la columna categorias
#-------------------------------------
# Corrección de errores ortográficos mediante un diccionario
categorias_map = {
    'office supplies': 'Office Supplies',
    'office supplis': 'Office Supplies',
    'oficce suplis': 'Office Supplies',
    'furniture': 'Furniture',
    'forniture': 'Furniture',
    'technology': 'Technology',
    'tecnology': 'Technology',
}

datos_ordenes['category'] = (datos_ordenes['category'].str.lower().map(categorias_map).fillna(datos_ordenes['category'].str.title()))
datos_ordenes['sub_category'] = datos_ordenes['sub_category'].str.title()

#Limpiar columna sales
#-------------------------------------
# Remover símbolos financieros para convertir ventas a valores numéricos
datos_ordenes['sales'] = (
    datos_ordenes['sales']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.replace('$', '', regex=False)
    .str.strip()
)
datos_ordenes['sales'] = pd.to_numeric(datos_ordenes['sales'], errors='coerce')

# Convertir tipos de datos a int y float según corresponda
datos_ordenes['quantity'] = datos_ordenes['quantity'].astype(int)
datos_ordenes['discount'] = datos_ordenes['discount'].astype(float)
datos_ordenes['profit'] = datos_ordenes['profit'].astype(float)
datos_ordenes['shipping_cost'] = datos_ordenes['shipping_cost'].astype(float)
datos_ordenes['year'] = datos_ordenes['year'].astype(int)

#Limpieza de Nulos
#-------------------------------------
# Unificar diferentes representaciones de "vacío" a NaN
datos_ordenes.replace(['nan', 'NaN', 'None', 'null', 'N/A', '?', 'TBD', '-'], np.nan, inplace=True)

# Imputar valores nulos en métricas numéricas utilizando la mediana
columnas_num_imputables = ['quantity', 'discount', 'year']
for col in columnas_num_imputables:
    if col in datos_ordenes.columns and datos_ordenes[col].isnull().sum() > 0:
        datos_ordenes[col] = datos_ordenes[col].fillna(datos_ordenes[col].median())

#Tipo de dato de fecha
#-------------------------------------
# Convertir textos a formato de fecha estándar
datos_ordenes['order_date'] = pd.to_datetime(datos_ordenes['order_date'], dayfirst=True, errors='coerce')
datos_ordenes['ship_date'] = pd.to_datetime(datos_ordenes['ship_date'], dayfirst=True, errors='coerce')

# Nueva columna: process_time (Diferencia de días entre fecha de envío y fecha de pedido)
datos_ordenes['process_time'] = (datos_ordenes['ship_date'] - datos_ordenes['order_date']).dt.days

# Nueva columna: limit_day (Limite de fecha por el tipo de prioridad)
limites_map = {
    'Critical': 0,
    'High': 2,
    'Medium': 4,
    'Low': 6
}
datos_ordenes['limit_day'] = datos_ordenes['order_priority'].map(limites_map) #Asignación del limite de dias, dependiendo de la prioridad de la orden
datos_ordenes['diference'] = datos_ordenes['process_time'] - datos_ordenes['limit_day'] #Diferencia entre el tiempo de procesamiento con el limite establecido, el retraso real comparado con el limite establecido

print("\nLimpieza de formato Columna Country")
#-------------------------------------
# Normalizar países quitando espacios extra y puntos
s = (datos_ordenes['country'].str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip("."))

# Clave base en minúsculas y sin caracteres especiales para agrupar similitudes
key = s.str.lower().str.replace(r"[^a-z]", "", regex=True)

# Limpieza de columna país
combinado = s[(s != s.str.upper()) & (s != s.str.lower())] #Busca los nombres de paises con capitalizacion mezclada (que tenga mayusculas y minusculas)
normalizar = combinado.groupby(key[combinado.index]).agg(lambda x: x.value_counts().index[0]) #Agrupa los paises obtenidos anteriormente por una clave

datos_ordenes['country'] = key.map(normalizar) #Normaliza los nombres de los paises

print(f'Cantidad de paises vacios: {datos_ordenes["country"].isna().sum()}')
print(f'Cantidad de paises unicos: {datos_ordenes["country"].nunique()}')
print("\nValidar nombres de países: ", sorted(datos_ordenes['country'].unique()))
print(f'Cantidad de paises con espacios en medio: {datos_ordenes["country"].str.contains(" ").sum()}')

# Validar limpieza aplicada
c = datos_ordenes['country']
print('Validación de limpieza')
print(f'Cantidad de paises con espacios adicionales: {c.str.contains(r"^\s|\s$|  ").sum()}')
print(f'Cantidad de paises con capitalizacion unicamente de mayusculas o minusculas: {((c == c.str.upper()) | (c == c.str.lower())).sum()}')
print("\nValidar países cuyo nombre tiene espacios:", sorted([x for x in c.unique() if ' ' in x]))
print(f'Cantidad de paises unicos con espacios en medio: {len([x for x in c.unique() if " " in x])}')

print("\nCorrección de valores nulos en discount")
#-------------------------------------
# Convertir descuentos negativos a positivos
datos_ordenes['discount'] = abs(datos_ordenes['discount'])
print(f'Cantidad de valores de discount negativos: {datos_ordenes[datos_ordenes["discount"] < 0]["discount"].count()}')

print("\n Detección de Valores Atípicos de Process Time por Categoría (Cuartiles Q1 y Q3)")
#-------------------------------------
# Calcular Q1 (25%), Q3 (75%) e IQR de process_time por cada categoría para detectar atípicos
q1 = datos_ordenes.groupby('order_priority')['process_time'].transform(lambda x: x.quantile(0.25))
q3 = datos_ordenes.groupby('order_priority')['process_time'].transform(lambda x: x.quantile(0.75))
iqr = q3 - q1

limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

print('Limites inferior y superior por tipo de prioridad')
for prioridad, grupo in datos_ordenes.groupby('order_priority'):
    indices = grupo.index

    print(f'\nPrioridad: {prioridad}')
    print(f'Límite inferior: {limite_inferior.loc[indices].iloc[0]}')
    print(f'Límite superior: {limite_superior.loc[indices].iloc[0]}')

# Identificar cuáles filas son valores atípicos
es_atipico = (datos_ordenes['process_time'] < limite_inferior) | (datos_ordenes['process_time'] > limite_superior)

# Separar en dos DataFrames conservando estrictamente el orden original de las filas
datos_ordenes_limpios = datos_ordenes[~es_atipico].copy()
datos_ordenes_atipicos = datos_ordenes[es_atipico].copy()

# Convertir columnas objeto a string en ambos DataFrames
for df_temp in [datos_ordenes_limpios, datos_ordenes_atipicos]:
    columnas_objeto = df_temp.select_dtypes(include=['object']).columns
    for col in columnas_objeto:
        df_temp[col] = df_temp[col].astype('string')

# Mostrar vista previa
print('\n                     Registros limpios')
display(datos_ordenes_limpios)

print('\n                     Registros atipicos')
display(datos_ordenes_atipicos)

# Exportar y descargar ambos archivos CSV
datos_ordenes_limpios.to_csv('datos_ordenes_limpios.csv', index=False)
files.download('datos_ordenes_limpios.csv')

datos_ordenes_atipicos.to_csv('datos_ordenes_atipicos.csv', index=False)
files.download('datos_ordenes_atipicos.csv')


print("\n Auditoria de validacion de datos")
#-------------------------------------
# Usamos el DataFrame con los datos limpios
df_auditoria = datos_ordenes_limpios

print("\n            Auditoria de limpieza de datos               ")

# 1. Dimensiones generales
print(f"\nDimensiones totales: {df_auditoria.shape[0]:,} filas x {df_auditoria.shape[1]} columnas")

# 2. Verificación de Nulos
print("\n 1. Valoración de valores nulos ")
nulos = df_auditoria.isnull().sum()
nulos_filtrados = nulos[nulos > 0]

if len(nulos_filtrados) == 0:
    print("\nNo se encontraron valores nulos en ninguna columna.")
else:
    print("\nSe encontraron valores nulos en las siguientes columnas:")
    print(nulos_filtrados)

# 3. Resumen de Tipos de Datos y Valores Únicos
print("\n 2. Identificación de tipos de datos y valores únicos")
auditoria_df = pd.DataFrame({
    'Tipo de Dato': df_auditoria.dtypes.astype(str),
    'Valores Únicos': df_auditoria.nunique(),
    'Nulos Totales': df_auditoria.isnull().sum(),
    'Ejemplo Valor': [
        df_auditoria[col].dropna().iloc[0] if not df_auditoria[col].dropna().empty else 'N/A'
        for col in df_auditoria.columns
    ],
})
display(auditoria_df)

# 4. Verificación específica de Categorías
print("\n 3. Verificación de la limpieza de base de datos ")
print(f"  Categorías estandarizadas ({df_auditoria['category'].nunique()}):")
print(f"  {df_auditoria['category'].unique().tolist()}")

print(f"\n Subcategorías ({df_auditoria['sub_category'].nunique()}):")
print(f"  {df_auditoria['sub_category'].unique().tolist()}")

print(f"\n Modos de Envío ({df_auditoria['ship_mode'].nunique()}):")
print(f"  {df_auditoria['ship_mode'].unique().tolist()}")

print(f"\n Segmentos ({df_auditoria['segment'].nunique()}):")
print(f"  {df_auditoria['segment'].unique().tolist()}")

# 5. Rango de Fechas
print("\n 4. Análisis de datos de fechas ")
print(
    f" Fecha Mínima Pedido: {df_auditoria['order_date'].min().strftime('%Y-%m-%d')} |"
    f" Máxima: {df_auditoria['order_date'].max().strftime('%Y-%m-%d')}"
)
print(
    f" Fecha Mínima Envío:  {df_auditoria['ship_date'].min().strftime('%Y-%m-%d')} |"
    f" Máxima: {df_auditoria['ship_date'].max().strftime('%Y-%m-%d')}"
)

print("\n                 Fin de la auditoría                ")


 Cargar base datos


Saving SuperStoreOrders_dirty.csv to SuperStoreOrders_dirty (1).csv

                     Valores nulos                         
order_id          0
order_date        0
ship_date         0
ship_mode         0
customer_name     0
segment           0
state             0
country           0
market            0
region            0
product_id        0
category          0
sub_category      0
product_name      0
sales             0
quantity          0
discount          0
profit            0
shipping_cost     0
order_priority    0
year              0
dtype: int64
                     Valores unicos de paises                           
Cantidad de paises unicos: 147
Listado de paises: 
['Afghanistan', 'Albania', 'Algeria', 'Angola', 'Argentina', 'Armenia', 'Australia', 'Austria', 'Azerbaijan', 'Bahrain', 'Bangladesh', 'Barbados', 'Belarus', 'Belgium', 'Benin', 'Bolivia', 'Bosnia and Herzegovina', 'Brazil', 'Bulgaria', 'Burundi', 'Cambodia', 'Cameroon', 'Canada', 'Central African Republic', 'Cha

,order_id,order_date,ship_date,ship_mode,customer_name,segment,state,country,market,region,...,sales,quantity,discount,profit,shipping_cost,order_priority,year,process_time,limit_day,diference
0,AG-2011-2040,2011-01-01,2011-01-06,Standard Class,Toby Braunhardt,Consumer,Constantine,Algeria,Africa,Africa,...,408,2,0.0,106.1400,35.46,Medium,2011,5,4,1
2,HU-2011-1220,2011-01-01,2011-01-05,Second Class,Annie Thurman,Consumer,Budapest,Hungary,EMEA,EMEA,...,66,4,0.0,29.6400,8.17,High,2011,4,2,2
3,IT-2011-3647632,2011-01-01,2011-01-05,Second Class,Eugene Moren,Home Office,Stockholm,Sweden,EU,North,...,45,3,0.5,-26.0550,4.82,High,2011,4,2,2
6,CA-2011-1510,2011-01-02,2011-01-06,Standard Class,Magdelene Morse,Consumer,Ontario,Canada,Canada,Canada,...,314,1,0.0,3.1200,24.10,Medium,2011,4,4,0
7,IN-2011-79397,2011-01-03,2011-01-03,Same Day,Kean Nguyen,Corporate,New South Wales,Australia,APAC,Oceania,...,276,1,0.1,110.4120,125.32,Critical,2011,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51284,MX-2014-108574,2014-12-31,2015-01-04,Standard Class,Julia Barnett,Home Office,Tamaulipas,Mexico,LATAM,North,...,17,3,0.0,0.6600,1.32,Medium,2014,4,4,0
51285,CA-2014-115427,2014-12-31,2015-01-04,Standard Class,Erica Bern,Corporate,California,United States,US,West,...,14,2,0.2,4.5188,0.89,Medium,2014,4,4,0
51286,MO-2014-2560,2014-12-31,2015-01-05,Standard Class,Liz Preis,Consumer,Souss-Massa-Dra�,Morocco,Africa,Africa,...,4,1,0.0,0.4200,0.49,Medium,2014,5,4,1
51288,MX-2014-114783,2014-12-31,2015-01-06,Standard Class,Tamara Dahlen,Consumer,Chihuahua,Mexico,LATAM,North,...,7,1,0.0,0.5600,0.20,Medium,2014,6,4,2



                     Registros atipicos


,order_id,order_date,ship_date,ship_mode,customer_name,segment,state,country,market,region,...,sales,quantity,discount,profit,shipping_cost,order_priority,year,process_time,limit_day,diference
1,IN-2011-47883,2011-01-01,2011-01-08,Standard Class,Joseph Holt,Consumer,New South Wales,Australia,APAC,Oceania,...,120,3,0.1,36.036,9.72,Medium,2011,7,4,3
4,IN-2011-47883,2011-01-01,2011-01-08,Standard Class,Joseph Holt,Consumer,New South Wales,Australia,APAC,Oceania,...,114,5,0.1,37.770,4.70,Medium,2011,7,4,3
5,IN-2011-47883,2011-01-01,2011-01-08,Standard Class,Joseph Holt,Consumer,New South Wales,Australia,APAC,Oceania,...,55,2,0.1,15.342,1.80,Medium,2011,7,4,3
29,MX-2011-158771,2011-01-04,2011-01-11,Standard Class,Beth Thompson,Home Office,Granma,Cuba,LATAM,Caribbean,...,186,6,0.0,3.600,16.39,Medium,2011,7,4,3
33,US-2011-136007,2011-01-04,2011-01-11,Standard Class,Beth Thompson,Home Office,Alagoas,Brazil,LATAM,South,...,74,6,0.6,-107.856,7.04,Medium,2011,7,4,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51263,MX-2014-110527,2014-12-31,2015-01-02,Second Class,Charlotte Melton,Consumer,Managua,Nicaragua,LATAM,Central,...,136,3,0.0,10.860,6.00,Medium,2014,2,4,-2
51264,MX-2014-110527,2014-12-31,2015-01-02,Second Class,Charlotte Melton,Consumer,Managua,Nicaragua,LATAM,Central,...,72,8,0.0,6.400,5.64,Medium,2014,2,4,-2
51270,IN-2014-23754,2014-12-31,2015-01-07,Standard Class,Kalyca Meade,Corporate,Guangdong,China,APAC,North Asia,...,67,5,0.0,24.750,4.06,Medium,2014,7,4,3
51283,IN-2014-23754,2014-12-31,2015-01-07,Standard Class,Kalyca Meade,Corporate,Guangdong,China,APAC,North Asia,...,79,3,0.0,25.380,1.41,Medium,2014,7,4,3


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


 Auditoria de validacion de datos

            Auditoria de limpieza de datos               

Dimensiones totales: 46,574 filas x 24 columnas

 1. Valoración de valores nulos 

No se encontraron valores nulos en ninguna columna.

 2. Identificación de tipos de datos y valores únicos


,Tipo de Dato,Valores Únicos,Nulos Totales,Ejemplo Valor
order_id,string,22771,0,AG-2011-2040
order_date,datetime64[ns],1427,0,2011-01-01 00:00:00
ship_date,datetime64[ns],1462,0,2011-01-06 00:00:00
ship_mode,string,4,0,Standard Class
customer_name,string,795,0,Toby Braunhardt
segment,string,3,0,Consumer
state,string,1079,0,Constantine
country,string,146,0,Algeria
market,string,7,0,Africa
region,string,13,0,Africa



 3. Verificación de la limpieza de base de datos 
  Categorías estandarizadas (3):
  ['Office Supplies', 'Technology', 'Furniture']

 Subcategorías (17):
  ['Storage', 'Paper', 'Machines', 'Appliances', 'Copiers', 'Chairs', 'Tables', 'Bookcases', 'Phones', 'Accessories', 'Labels', 'Art', 'Supplies', 'Envelopes', 'Fasteners', 'Binders', 'Furnishings']

 Modos de Envío (4):
  ['Standard Class', 'Second Class', 'Same Day', 'First Class']

 Segmentos (3):
  ['Consumer', 'Home Office', 'Corporate']

 4. Análisis de datos de fechas 
 Fecha Mínima Pedido: 2011-01-01 | Máxima: 2014-12-31
 Fecha Mínima Envío:  2011-01-03 | Máxima: 2015-01-06

                 Fin de la auditoría                
